# Lid-driven cavity by finite differences (streamfunction--vorticity)

A traditional CFD solution of the classic **lid-driven cavity** at $Re=100$: a unit
square filled with fluid, the top wall (the *lid*) sliding at unit speed, the other
three walls stationary (no-slip). We solve the 2-D incompressible Navier--Stokes
equations in **streamfunction--vorticity** form, which eliminates the pressure and
enforces incompressibility exactly:
$$\nabla^2\psi=-\omega,\qquad
\frac{\partial\omega}{\partial t}+u\,\omega_x+v\,\omega_y=\frac{1}{Re}\nabla^2\omega,
\qquad u=\psi_y,\ v=-\psi_x .$$

**The CFD pipeline in miniature.** *Preprocessing*: lay a uniform $129\times129$ grid.
*Solving*: at each step, (i) relax the Poisson equation $\nabla^2\psi=-\omega$ for the
streamfunction, (ii) recover the velocities, (iii) set the wall vorticity from Thom's
formula, (iv) march the vorticity transport equation one explicit step. Iterate to
steady state. *Postprocessing*: streamlines, centreline profiles, and comparison with
the Ghia, Ghia \& Shin (1982) benchmark. Runs in well under a minute on a CPU.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':120,'font.size':15,'axes.titlesize':14,
    'axes.labelsize':15,'xtick.labelsize':13,'ytick.labelsize':13,'legend.fontsize':12})
np.random.seed(0)
RE = 100.0

In [ ]:
# --- The solver: streamfunction-vorticity finite differences ---
def cavity_fd(n=129, max_iter=200000, tol=1e-7):
    h  = 1.0/(n-1)
    psi = np.zeros((n, n)); om = np.zeros((n, n))
    dt = min(0.25*h*h*RE, 0.5*h)            # explicit-stability time step
    hist = []
    t0 = time.perf_counter()
    for it in range(max_iter):
        for _ in range(30):                 # Poisson: lap(psi) = -om  (Jacobi sweeps)
            psi[1:-1,1:-1] = 0.25*(psi[2:,1:-1]+psi[:-2,1:-1]
                                   +psi[1:-1,2:]+psi[1:-1,:-2] + h*h*om[1:-1,1:-1])
        u = np.zeros((n,n)); v = np.zeros((n,n))
        u[1:-1,1:-1] =  (psi[1:-1,2:]-psi[1:-1,:-2])/(2*h)     # u =  d psi/dy
        v[1:-1,1:-1] = -(psi[2:,1:-1]-psi[:-2,1:-1])/(2*h)     # v = -d psi/dx
        u[:,-1] = 1.0                         # moving lid at y = 1
        om_new = om.copy()                    # wall vorticity (Thom's formula)
        om_new[1:-1,0]  = -2*psi[1:-1,1]/h**2
        om_new[1:-1,-1] = -2*psi[1:-1,-2]/h**2 - 2.0/h
        om_new[0,1:-1]  = -2*psi[1,1:-1]/h**2
        om_new[-1,1:-1] = -2*psi[-2,1:-1]/h**2
        adv = (u[1:-1,1:-1]*(om[2:,1:-1]-om[:-2,1:-1])/(2*h)
               + v[1:-1,1:-1]*(om[1:-1,2:]-om[1:-1,:-2])/(2*h))
        lap = (om[2:,1:-1]+om[:-2,1:-1]+om[1:-1,2:]+om[1:-1,:-2]-4*om[1:-1,1:-1])/h**2
        om_new[1:-1,1:-1] = om[1:-1,1:-1] + dt*(lap/RE - adv)
        d = np.max(np.abs(om_new-om))/(np.max(np.abs(om_new))+1e-12); om = om_new
        hist.append(d)
        if d < tol and it > 1000:
            break
    print(f'converged at iter {it} in {time.perf_counter()-t0:.1f}s  (residual {d:.1e})')
    return psi, u, v, om, np.array(hist)

psi, u, v, om, hist = cavity_fd()
n = u.shape[0]; g = np.linspace(0, 1, n)
u_cl = u[n//2, :]      # u along the vertical centreline x = 0.5
v_cl = v[:, n//2]      # v along the horizontal centreline y = 0.5

In [ ]:
# --- Ghia, Ghia & Shin (1982) benchmark for Re=100 ---
y_ghia = np.array([1.0,0.9766,0.9688,0.9609,0.9531,0.8516,0.7344,0.6172,0.5,
                   0.4531,0.2813,0.1719,0.1016,0.0703,0.0625,0.0547,0.0])
u_ghia = np.array([1.0,0.84123,0.78871,0.73722,0.68717,0.23151,0.00332,-0.13641,
                   -0.20581,-0.2109,-0.15662,-0.1015,-0.06434,-0.04775,-0.04192,-0.03717,0.0])
x_ghia = np.array([1.0,0.9688,0.9609,0.9531,0.9453,0.9063,0.8594,0.8047,0.5,
                   0.2344,0.2266,0.1563,0.0938,0.0781,0.0703,0.0625,0.0])
v_ghia = np.array([0.0,-0.05906,-0.07391,-0.08864,-0.10313,-0.16914,-0.22445,-0.24533,
                   0.05454,0.17527,0.17507,0.16077,0.12317,0.1089,0.10091,0.09233,0.0])

u_at = np.interp(y_ghia, g, u_cl)   # FD result sampled at Ghia's points
v_at = np.interp(x_ghia, g, v_cl)
err_u = np.sqrt(np.mean((u_at-u_ghia)**2)/np.mean(u_ghia**2))
err_v = np.sqrt(np.mean((v_at-v_ghia)**2)/np.mean(v_ghia**2))
print(f'u(0.5,0.5) = {np.interp(0.5,g,u_cl):+.4f}   (Ghia -0.20581)')
print(f'rel L2 vs Ghia:  u {err_u:.3f},  v {err_v:.3f}')
# primary vortex centre = extremum of the streamfunction
i0, j0 = np.unravel_index(np.argmax(np.abs(psi)), psi.shape)
print(f'primary vortex centre (x,y) = ({g[i0]:.3f}, {g[j0]:.3f}),  psi = {psi[i0,j0]:+.4f}')
print(f'               (Ghia 1982: (0.6172, 0.7344),  psi = -0.103)')

In [ ]:
# --- Postprocessing: streamlines, centreline profiles vs Ghia, convergence ---
fig, ax = plt.subplots(2, 2, figsize=(10.5, 9.2)); a = ax.ravel()
spd = np.sqrt(u**2 + v**2)
c = a[0].contourf(g, g, spd.T, 21, cmap='viridis')
a[0].streamplot(g, g, u.T, v.T, color='w', density=1.2, linewidth=0.6)
plt.colorbar(c, ax=a[0]); a[0].set_title('streamlines over speed'); a[0].set_aspect('equal')
a[0].set_xlabel('x'); a[0].set_ylabel('y'); a[0].set_xlim(0,1); a[0].set_ylim(0,1)

a[1].plot(u_cl, g, 'b-', lw=2, label='FD (129$^2$)')
a[1].plot(u_ghia, y_ghia, 'ko', ms=5, label='Ghia et al. 1982')
a[1].set_xlabel('u(0.5, y)'); a[1].set_ylabel('y'); a[1].legend(); a[1].grid(alpha=.3)
a[1].set_title('vertical centreline')

a[2].plot(g, v_cl, 'b-', lw=2, label='FD (129$^2$)')
a[2].plot(x_ghia, v_ghia, 'ko', ms=5, label='Ghia et al. 1982')
a[2].set_xlabel('x'); a[2].set_ylabel('v(x, 0.5)'); a[2].legend(); a[2].grid(alpha=.3)
a[2].set_title('horizontal centreline')

a[3].semilogy(hist, 'r-', lw=1.5)
a[3].set_xlabel('iteration'); a[3].set_ylabel('vorticity residual')
a[3].set_title('convergence to steady state'); a[3].grid(alpha=.3, which='both')
plt.tight_layout()
plt.savefig('cavity_cfd_fdm.png', bbox_inches='tight'); print('saved figure')

**What to take away.** The finite-difference solution lands on the Ghia benchmark to
within a percent or two at $129^2$, resolves the primary recirculating vortex and the two
weak secondary vortices in the bottom corners, and converges monotonically to steady
state. This is the *classical* answer against which the physics-informed cavity of the
Capstone chapter is measured -- and it is fast, accurate, and mesh-bound, exactly the
trade-off Chapter~4 laid out.